# Full training runs: our ViT, 0 vs 4 registers (Google Colab)

Trains ViT-Tiny on Tiny-ImageNet for 50 epochs, five times:

| Run | Registers | Augmentation | Training images |
|---|---|---|---|
| `tiny_reg0` | 0 | basic (crop + flip) | 100,000 |
| `tiny_reg4` | 4 | basic | 100,000 |
| `tiny_reg0_strong` | 0 | strong (+ RandAugment, random erasing, MixUp/CutMix) against overfitting | 100,000 |
| `tiny_reg4_strong` | 4 | strong | 100,000 |
| `tiny_reg0_strong_50pct` | 0 | strong | 50,000 (does the amount of data matter?) |

Then it makes a results table and graph.

**Time:** about 85 minutes per full run, 45 for the 50% run: about 3.5 hours for the three new runs. Runs that already finished are skipped instantly.

**Before running:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.
Keep this tab open and your laptop awake while it runs.

**If Colab disconnects:** just do `Runtime → Run all` again. Progress is saved to your
Google Drive after every epoch, so training continues from where it stopped.

## 0. Setup: get the code

In [ ]:
import os

REPO_URL = "https://github.com/recommended131225435/vit-registers-reproduction.git"

if os.path.exists("../src") and not os.path.exists("src"):  # opened from inside notebooks/
    %cd ..
if not os.path.exists("src"):                                  # fresh Colab session
    !git clone -q {REPO_URL} repo
    %cd repo
else:
    !git pull -q                                               # make sure the code is up to date
!pip install -q -r requirements.txt

import torch
from IPython.display import Image, Markdown, display
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime to T4!)")

## 1. Connect Google Drive
The trained models and logs are saved here, so nothing is lost if Colab disconnects. Click **Allow** when asked.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

RUN_DIR = "/content/drive/MyDrive/vit-registers-runs"
os.makedirs(RUN_DIR, exist_ok=True)
print("Runs will be saved in", RUN_DIR)

## 2. Basic augmentation: without and with registers
Already finished runs print `Resuming from epoch 50` and are skipped.

In [ ]:
!python -m src.train --size tiny --registers 0 --epochs 50 --resume --out {RUN_DIR}/tiny_reg0
!python -m src.train --size tiny --registers 4 --epochs 50 --resume --out {RUN_DIR}/tiny_reg4

## 3. Strong augmentation (against overfitting): without registers (about 80 min)

In [ ]:
!python -m src.train --size tiny --registers 0 --aug strong --epochs 50 --resume --out {RUN_DIR}/tiny_reg0_strong

## 4. Strong augmentation: with 4 registers (about 80 min)

In [ ]:
!python -m src.train --size tiny --registers 4 --aug strong --epochs 50 --resume --out {RUN_DIR}/tiny_reg4_strong

## 5. Strong augmentation on half the data (about 45 min)
Same as step 3, but trained on a fixed random 50,000 of the 100,000 images. Comparing it with step 3 shows how much the amount of data matters.

In [ ]:
!python -m src.train --size tiny --registers 0 --aug strong --train-subset 50000 --epochs 50 --resume --out {RUN_DIR}/tiny_reg0_strong_50pct

## 6. Results table and graph

In [ ]:
!python -m scripts.summarise_training --runs {RUN_DIR}/tiny_reg0 {RUN_DIR}/tiny_reg4 {RUN_DIR}/tiny_reg0_strong {RUN_DIR}/tiny_reg4_strong {RUN_DIR}/tiny_reg0_strong_50pct

display(Markdown(open("results/runs/full_training_summary.md").read().split("![")[0]))
display(Image("results/figures/full_training_runs.png"))

## 7. Download the results
The zip holds the logs, the graph and the updated README (not the model weights; those stay on your Google Drive).

In [ ]:
!zip -qr full_training_results.zip results/runs results/figures/full_training_runs.png README.md

from google.colab import files
files.download("full_training_results.zip")